# So sánh U-Net / SegNet / DuckNet trên Kvasir-SEG

**Nguyên tắc so sánh công bằng:**
- Cùng `DATA_CFG` (seed=42) → cùng 700 ảnh train / 300 ảnh test cho cả 3 model
- `torch.manual_seed(42)` trước mỗi model → weight khởi tạo tái lập được
- Cùng `epochs`, `lr`, `optimizer`, `loss`, `batch_size`
- **Không** chạy cell override riêng của từng model khi muốn so sánh công bằng

In [ ]:
from pathlib import Path
import sys

try:
    from IPython import get_ipython
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'trains':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import torch
import matplotlib.pyplot as plt
import pandas as pd

from utils.data       import PolypDataConfig, build_kvasir_dataloaders
from utils.train      import PolypTrainConfig, train_polyp_segmentation, load_checkpoint, evaluate_model
from utils.evaluation import report_tuned_metrics
from utils.losses     import BCEDiceLoss
from models.unet      import UNet
from models.segnet    import SegNet
from models.ducknet   import DuckNet

# ── Cấu hình chung cho cả 3 model ──────────────────────────────────────────
DATA_CFG = PolypDataConfig(image_size=256, batch_size=4, seed=42)
FAIR_CFG = dict(
    epochs=100,
    learning_rate=3e-4,   # 1e-3 quá cao cho AdamW → model dao động, không hội tụ sâu
    weight_decay=1e-4,
    lr_scheduler='cosine',
    cosine_min_lr=1e-6,
    warmup_epochs=5,      # 5 epoch đầu warm-up tránh loss spike
    checkpoint_policy='best',  # lưu checkpoint tốt nhất thay vì last
    eval_each_epoch=True,
)

# ── Load data 1 lần duy nhất, dùng chung cho cả 3 model ────────────────────
loaders, split_summary, DATA_ROOT = build_kvasir_dataloaders(PROJECT_ROOT, DATA_CFG)
print(f'Data: {DATA_ROOT}')
print(f'Split → train: {split_summary["train"]} | test: {split_summary["test"]}')

## 1. U-Net

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_unet = UNet(in_channels=3, num_classes=1)
cfg_unet   = PolypTrainConfig(
    model_name='UNet',
    save_path=PROJECT_ROOT / 'checkpoints' / 'unet_fair_last.pth',
    **FAIR_CFG,
)

history_unet, ckpt_unet = train_polyp_segmentation(
    model_unet, loaders, DATA_CFG, cfg_unet, split_summary, DATA_ROOT
)
print(f'Checkpoint lưu tại: {ckpt_unet}')

## 2. SegNet

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_segnet = SegNet(in_channels=3, num_classes=1)
cfg_segnet   = PolypTrainConfig(
    model_name='SegNet',
    save_path=PROJECT_ROOT / 'checkpoints' / 'segnet_fair_last.pth',
    **FAIR_CFG,
)

history_segnet, ckpt_segnet = train_polyp_segmentation(
    model_segnet, loaders, DATA_CFG, cfg_segnet, split_summary, DATA_ROOT
)
print(f'Checkpoint lưu tại: {ckpt_segnet}')

## 3. DuckNet

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_ducknet = DuckNet(in_channels=3, num_classes=1, base_channels=32, dropout=0.1)
cfg_ducknet   = PolypTrainConfig(
    model_name='DuckNet',
    save_path=PROJECT_ROOT / 'checkpoints' / 'ducknet_fair_last.pth',
    **FAIR_CFG,
)

history_ducknet, ckpt_ducknet = train_polyp_segmentation(
    model_ducknet, loaders, DATA_CFG, cfg_ducknet, split_summary, DATA_ROOT
)
print(f'Checkpoint lưu tại: {ckpt_ducknet}')

## 4. Đánh giá toàn diện — 6 độ đo

In [ ]:
criterion = BCEDiceLoss()
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

results = {}
for name, ModelClass, kwargs, ckpt in [
    ('UNet',    UNet,    {'in_channels': 3, 'num_classes': 1},                          ckpt_unet),
    ('SegNet',  SegNet,  {'in_channels': 3, 'num_classes': 1},                          ckpt_segnet),
    ('DuckNet', DuckNet, {'in_channels': 3, 'num_classes': 1, 'base_channels': 32, 'dropout': 0.1}, ckpt_ducknet),
]:
    m = load_checkpoint(ModelClass(**kwargs), ckpt).to(device)
    r = evaluate_model(m, loaders['test'], criterion, device)
    results[name] = r
    print(f'{name} → Acc={r["Accuracy"]:.4f} | Dice={r["Dice"]:.4f} | IoU={r["IoU"]:.4f} | Prec={r["Precision"]:.4f} | Rec={r["Recall"]:.4f} | F1={r["F1"]:.4f}')

# Bảng tổng hợp
df = pd.DataFrame(results, index=['Accuracy', 'Dice', 'IoU', 'Precision', 'Recall', 'F1', 'loss', 'iou']).T
df = df[['Accuracy', 'Dice', 'IoU', 'Precision', 'Recall', 'F1']].round(4)
print('\n=== Bảng so sánh 3 mô hình ===')
print(df.to_string())

## 5. Biểu đồ quá trình học

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors   = {'UNet': 'steelblue', 'SegNet': 'darkorange', 'DuckNet': 'seagreen'}
histories = {'UNet': history_unet, 'SegNet': history_segnet, 'DuckNet': history_ducknet}

for name, h in histories.items():
    c = colors[name]
    axes[0].plot(h['train_loss'], label=f'{name} train', color=c, linestyle='--', alpha=0.6)
    axes[0].plot(h['test_loss'],  label=f'{name} test',  color=c)
    axes[1].plot(h['train_iou'],  label=f'{name} train', color=c, linestyle='--', alpha=0.6)
    axes[1].plot(h['test_iou'],   label=f'{name} test',  color=c)

axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()
axes[1].set_title('IoU');  axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('IoU');  axes[1].legend()

plt.suptitle('So sánh quá trình học — U-Net / SegNet / DuckNet', fontsize=13)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'checkpoints' / 'training_curves.png', dpi=150)
plt.show()

## 6. Trực quan hoá dự đoán (3 ảnh mẫu)

In [ ]:
import random, torch
import matplotlib.pyplot as plt

test_dataset = loaders['test'].dataset
sample_indices = random.Random(0).sample(range(len(test_dataset)), 3)

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

model_list = [
    ('UNet',    load_checkpoint(UNet(in_channels=3, num_classes=1), ckpt_unet).to(device)),
    ('SegNet',  load_checkpoint(SegNet(in_channels=3, num_classes=1), ckpt_segnet).to(device)),
    ('DuckNet', load_checkpoint(DuckNet(in_channels=3, num_classes=1, base_channels=32, dropout=0.1), ckpt_ducknet).to(device)),
]
for m in model_list: m[1].eval()

fig, axes = plt.subplots(3, 5, figsize=(18, 11))
col_titles = ['Ảnh gốc', 'Ground Truth', 'U-Net', 'SegNet', 'DuckNet']
for ax, t in zip(axes[0], col_titles): ax.set_title(t, fontsize=11)

with torch.no_grad():
    for row, idx in enumerate(sample_indices):
        img, mask = test_dataset[idx]
        img_np = (img.permute(1,2,0).numpy() * std + mean).clip(0,1)

        axes[row][0].imshow(img_np);                               axes[row][0].axis('off')
        axes[row][1].imshow(mask.squeeze().numpy(), cmap='gray');  axes[row][1].axis('off')

        for col, (mname, model) in enumerate(model_list, start=2):
            pred = torch.sigmoid(model(img.unsqueeze(0).to(device)))
            pred_bin = (pred > 0.5).float().squeeze().cpu().numpy()
            axes[row][col].imshow(pred_bin, cmap='gray'); axes[row][col].axis('off')

plt.suptitle('Dự đoán phân đoạn polyp — 3 ảnh mẫu từ tập test', fontsize=13)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'checkpoints' / 'predictions_comparison.png', dpi=150)
plt.show()